# 1-2 Advanced: Evidence-based Model Choice 규칙의 문맥 실패를 확인하고, 데이터와 운영 조건을 tuple로 관리한 뒤 `recommend_start(*case)`로 풀어 전달했습니다. 불완전한 검증 근거에서는 승인과 재측정을 구분했습니다. 상태: 별도 심화 실습 완료. 

In [1]:
# 심화 실습 코드
# 키워드 일치 여부와 실제 label을 인덱스별로 대조해 실행 성공과 의미 성공을 구분합니다.
# 부정·인과 문맥을 잃는 규칙의 오분류만 수집하고 전체 서비스 품질로 과장하지 않습니다.
texts = ["환불이 아니라 교환을 원합니다", "로그인이 막혀 결제가 안 됩니다", "환불 부탁드립니다"]
labels = ["교환", "계정", "환불"]

def current_rule(text):
    # 단어 존재만 확인하므로 부정과 문장 전체 의미는 보존하지 못합니다.
    if "환불" in text:
        return "환불"
    if "결제" in text:
        return "결제"
    return "기타"

preds = [current_rule(text) for text in texts]
wrong = [i for i, (pred, label) in enumerate(zip(preds, labels)) if pred != label] # 확인하기
print("preds:", preds)
print("wrong_indices:", wrong)

preds: ['환불', '결제', '환불']
wrong_indices: [0, 1]


In [2]:

# 샘플 수·규칙 유지비·문맥 의존도를 각각 검사해 딥러닝 적용 여부를 한 조건으로 단정하지 않습니다.
# 입력 숫자는 팀의 수업용 기준과 비교하고 결과는 배포 승인이 아니라 다음 실험 권고로 제한합니다.
def recommend_start(raw_unstructured, labeled_count, stable_rule):
    # 명시적이고 안정적인 규칙은 데이터 양과 무관하게 먼저 보존합니다.
    if stable_rule:
        return "rule_first"
    # 원본 텍스트·이미지처럼 표현 설계가 어려우면서 검증 가능한 라벨이 있을 때만 후보로 올립니다.
    if raw_unstructured and labeled_count >= 5000:
        return "deep_learning_candidate"
    # 데이터가 적거나 구조화 입력이면 단순 기준선을 먼저 만들어 비교 기준을 남깁니다.
    return "simple_baseline_first"

cases = [(False, 400, True), (True, 28000, False), (True, 120, False)]
print([recommend_start(*case) for case in cases])

['rule_first', 'deep_learning_candidate', 'simple_baseline_first']


In [ ]:
# 심화 실습 코드
# 필수 증거가 모두 있는지 먼저 확인하고 누락된 latency·비용 자료는 임의 값으로 채우지 않습니다.
# 증거가 부족하면 reject가 아니라 remeasure를 반환해 모델 실패와 측정 미완료를 분리합니다.
candidates = {
    "A": {"accuracy": 0.884, "latency_ms": 3, "latency_runs": 3},
    "B": {"accuracy": 0.921, "latency_ms": 10, "latency_runs": 1},
    "C": {"accuracy": 0.908, "latency_ms": 13, "latency_runs": 3},
}
approved, remeasure, rejected = [], [], {}
for name, result in candidates.items():
    failed = []
    if result["accuracy"] < 0.90:
        failed.append("accuracy")
    if result["latency_ms"] > 12:
        failed.append("latency")
    if failed:
        rejected[name] = failed
    elif result["latency_runs"] < 3:
        remeasure.append(name)
    else:
        approved.append(name)
decision = max(approved, key=lambda n: candidates[n]["accuracy"]) if approved else "보류"
print("approved:", approved)
print("remeasure:", remeasure)
print("rejected:", rejected)
print("decision:", decision)